In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os
from pathlib import Path
import matplotlib as mpl
from matplotlib.offsetbox import AnchoredText
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from experiments.configs.classification_consts import MODELS
plt.rcParams['text.usetex'] = True
plt.rcParams.update({
    "text.usetex": True,              # Use TeX for text rendering
    "font.family": "serif",
    "hatch.color": "white"
})
%load_ext autoreload
%autoreload 2

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from src.PCS.classification.multi_class_jucal import MultiClassPCS_JUCAL
from experiments.configs.classification_configs import get_classification_datasets
from src.metrics.classification_metrics import get_uncertainties

In [3]:
DATASETS = ["data_yeast"]
RESULT_PATH = Path('../results/ablation/boot')

In [4]:
def get_dataset(X, y, features, sample_proportion):

    Xtrain, Xtest, ytrain, ytest = train_test_split(
        X[features], y, test_size=0.25, random_state=42
    )
    n_samples = int(np.floor(sample_proportion*len(Xtrain)))
    print(features)
    Xtrain = Xtrain.to_numpy()
    Xtest = Xtest.to_numpy()
    indices = np.random.choice(
        range(len(Xtrain)), size=n_samples, replace=False
    )
    return Xtrain[indices], Xtest, ytrain[indices], ytest

In [ ]:
dataset_key = 0
X, y, _, importance = get_classification_datasets(DATASETS[dataset_key])
num_classes = len(np.unique(y))
# sample_proportions = np.linspace(0.75, 1, 10)
sample_proportions = [1]

C1_mat = np.full(shape=(len(sample_proportions), len(importance)), fill_value=None)
C2_mat = np.full(shape=(len(sample_proportions), len(importance)), fill_value=None)
aleatoric_mat = np.full(shape=(len(sample_proportions), len(importance)), fill_value=None)
epistemic_mat = np.full(shape=(len(sample_proportions), len(importance)), fill_value=None)
NLL_mat = np.full(shape=(len(sample_proportions), len(importance)), fill_value=None)

for (i, sample_proportion) in enumerate(sample_proportions):
    # for j in range(len(importance)):
        j = len(importance)-1

        Xtrain, Xtest, ytrain, ytest = get_dataset(X, y, importance.iloc[0:j+1]["feature"], sample_proportion)
        

        pcs_JUCAL = MultiClassPCS_JUCAL(
            MODELS,
            num_bootstraps=500,
            n_classes=len(np.unique(y)),
            alpha=0.1,
            seed=42,
            top_k=1,
            save_path="./models",
            load_models=False,
            metric=log_loss
        )
        pcs_JUCAL.fit(Xtrain, ytrain)
        ensemble = pcs_JUCAL.ensemble(Xtest)

        C1_mat[i, j] = pcs_JUCAL.c1
        C2_mat[i, j] = pcs_JUCAL.c2

        Uepistemic, Ualeatoric = get_uncertainties(ensemble)
        epistemic_mat[i, j] = Uepistemic
        aleatoric_mat[i, j] = Ualeatoric 

        probas = np.nanmean(ensemble, axis=2)
        NLL_mat[i, j] = pcs_JUCAL.metric(ytest, probas, labels=range(num_classes))